In [ ]:
from pathlib import Path

import requests
import pandas as pd
import io
import os
import json

# ==============================================================================
# CONFIGURATION FOR MULTIPLE DATASETS
# ==============================================================================
TARGET_DATASETS = [
    {
        "file_name": "OECD_RD_GDP_2000_2025.csv",
        "agency_id": "OECD.STI.STP",
        "dataflow_id": "DSD_MSTI@DF_MSTI",
        "sdmx_query_key": "all",  # Small dataset, safe to download "all" and filter via Pandas
        "filters": {
            "UNIT_MEASURE": "PT_B1GQ",
            "MEASURE": "G"
        }
    },
    {
        "file_name": "OECD_Inflation_CPI_2000_2025.csv",
        "agency_id": "OECD.SDD.TPS",
        "dataflow_id": "DSD_PRICES@DF_PRICES_ALL",
        "sdmx_query_key": ".A.N.CPI.._T.N.GY",
        "filters": {} 
    },
    {
        "file_name": "OECD_Unemployment_Rate_2000_2025.csv",
        "agency_id": "OECD.ELS.SAE",
        "dataflow_id": "DSD_LFS@DF_LFS_INDIC",
        "sdmx_query_key": "all",  
        "filters": {
            "MEASURE": "UNE_RATE",    # Unemployment Rate
            "UNIT_MEASURE": "PT_LF_SUB", # % of Labour Force
            "SEX": "_T",              # Total (Male + Female)
            "AGE": "_T"               # Total Age
        }
    },
    {
        "file_name": "OECD_Productivity_Variation_2000_2025.csv",
        "agency_id": "OECD.SDD.TPS",
        "dataflow_id": "DSD_PDB@DF_PDB_LV",
        "sdmx_query_key": "all",  
        "filters": {
            "MEASURE": "GDPHRS"  # GDP per hour worked (Labour Productivity)
        }
    },
    {
        "file_name": "OECD_Tertiary_Education_2000_2025.csv",
        "agency_id": "OECD.EDU.IMEP",
        "dataflow_id": "DSD_EAG_LSO_EA@DF_LSO_NEAC_DISTR_EA",
        "sdmx_query_key": "._T.Y25T64.ISCED11A_5T8..........OBS...A",  
        "filters": {} # Left empty because the API query key handles all the filtering on the server
    },
    {   "file_name": "OECD_GDP_Growth_2000_2025.csv",
        "agency_id": "OECD.SDD.NAD",
        "dataflow_id": "DSD_NAMAIN1@DF_QNA_EXPENDITURE_GROWTH_OECD",
        "sdmx_query_key": "A.Y..S1.S1.B1GQ._Z._Z._Z.PC.L.GY.T0102",
        "filters": {}  # No Pandas-side filtering needed; server-side key is already precise
    }
]

START_PERIOD = "2000"
END_PERIOD = "2025"
VERSION = "all" # Uses the latest dataset version
PROJECT_ROOT = Path.cwd()
DATA_RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

headers = {
    "Accept": "application/vnd.sdmx.data+csv; charset=utf-8; version=2",
    "User-Agent": "DataExtractionScript/4.0 (Python/Requests)"
}

# ==============================================================================
# BATCH EXTRACTION PROCESS
# ==============================================================================

for dataset in TARGET_DATASETS:
    print(f"Processing: {dataset['file_name']}...")
    
    # --------------------------------------------------------------------------
    # FILE EXISTENCE CHECK (SKIP IF ALREADY DOWNLOADED)
    # --------------------------------------------------------------------------
    output_path = DATA_RAW_DIR / dataset['file_name']
    if output_path.exists():
        print(f"  -> File '{dataset['file_name']}' already exists. Skipping download.\n")
        continue

    cwd_path = Path(dataset['file_name'])
    if output_path.exists() or cwd_path.exists():
        print(f"  -> File already exists (either {output_path} or {cwd_path}). Skipping download.\n")
        continue

    # Build the specific URL using the server-side query key to prevent memory overload
    query_key = dataset.get('sdmx_query_key', 'all')
    url = f"https://sdmx.oecd.org/public/rest/data/{dataset['agency_id']},{dataset['dataflow_id']},{VERSION}/{query_key}"
    
    params = {
        "startPeriod": START_PERIOD,
        "endPeriod": END_PERIOD,
        "dimensionAtObservation": "AllDimensions",
        "format": "csvfilewithlabels"
    }
    
    response = requests.get(url, params=params, headers=headers)
    
    if response.status_code == 200:
        csv_stream = io.StringIO(response.text)
        df = pd.read_csv(csv_stream, low_memory=False)
        
        # Protect against completely empty API responses
        if 'OBS_VALUE' not in df.columns:
             print(f"  -> Error: No valid data returned by the API for {dataset['file_name']}.")
             continue
             
        df_clean = df.dropna(subset=['OBS_VALUE']).copy()
        
        # Apply local Pandas filters only if they are defined in the dictionary
        for column_name, filter_value in dataset.get('filters', {}).items():
            if column_name in df_clean.columns:
                df_clean = df_clean[df_clean[column_name] == filter_value]
            else:
                print(f"  -> Warning: Column '{column_name}' not found.")
        
        if df_clean.empty:
            print(f"  -> Error: Dataset is empty after applying filters.")
            
            debug_dict = {}
            for col in df.columns:
                if col not in ['OBS_VALUE', 'TIME_PERIOD', 'REF_AREA']:
                    debug_dict[col] = df[col].dropna().unique().tolist()
            
            with open("DEBUG_OECD_CODES.txt", "w") as f:
                f.write(json.dumps(debug_dict, indent=4))
                
            print("  -> Debug file generated: 'DEBUG_OECD_CODES.txt'. Please check the exact codes.\n")
            continue
        
        target_columns = ['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']
        
        if all(col in df_clean.columns for col in target_columns):
            df_tidy = df_clean[target_columns].rename(columns={
                'REF_AREA': 'Country',
                'TIME_PERIOD': 'Year',
                'OBS_VALUE': 'Value'
            })
            
            output_path = DATA_RAW_DIR / dataset['file_name']
            df_tidy = df_tidy.reset_index(drop=True)
            df_tidy.to_csv(output_path, index=False)
            
            print(f"  -> Success! Saved {len(df_tidy)} rows to {output_path}.\n")
        else:
            print(f"  -> Structural error: Missing SDMX columns in {dataset['file_name']}.\n")
    else:
        print(f"  -> HTTP Error {response.status_code} for {dataset['file_name']}.\n")

Processing: OECD_RD_GDP_2000_2025.csv...
  -> File 'OECD_RD_GDP_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Inflation_CPI_2000_2025.csv...
  -> File 'OECD_Inflation_CPI_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Unemployment_Rate_2000_2025.csv...
  -> File 'OECD_Unemployment_Rate_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Productivity_Variation_2000_2025.csv...
  -> File 'OECD_Productivity_Variation_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Public_Debt_GDP_2000_2025.csv...
  -> File 'OECD_Public_Debt_GDP_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Tertiary_Education_2000_2025.csv...
  -> File 'OECD_Tertiary_Education_2000_2025.csv' already exists. Skipping download.

Processing: OECD_GDP_Growth_2000_2025.csv...
  -> File 'OECD_GDP_Growth_2000_2025.csv' already exists. Skipping download.



In [2]:
import itertools
from pathlib import Path

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd()
DATA_RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

EUROSTAT_BASE = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"
EUROSTAT_HEADERS = {
    "Accept": "application/json",
    "User-Agent": "DataExtractionScript/4.0 (Python/Requests)",
}

EUROSTAT_DATASETS = [
    {
        "file_name": "Eurostat_Energy_Import_Dependency_2000_2025.csv",
        "dataset_code": "sdg_07_50",
        "params": {
            "format": "JSON",
            "lang": "EN",
            "sinceTimePeriod": "2000",
            "geoLevel": "country",
            "siec": "TOTAL",
            "unit": "PC",
        },
        "keep_columns": ["geo", "time", "OBS_VALUE"],
        "rename_map": {
            "geo": "Country",
            "time": "Year",
            "OBS_VALUE": "Value",
        },
    },
    {
        "file_name": "Eurostat_Public_Debt_GDP_2000_2025.csv",
        "dataset_code": "gov_10dd_edpt1",
        "params": {
            "format": "JSON",
            "lang": "EN",
            "sinceTimePeriod": "2000",
            "freq": "A",
            "unit": "PC_GDP",
            "sector": "S13",
            "na_item": "GD",
            "geoLevel": "country",
        },
        "keep_columns": ["geo", "time", "OBS_VALUE"],
        "rename_map": {
            "geo": "Country",
            "time": "Year",
            "OBS_VALUE": "Value",
        },
    },
]


def inspect_eurostat_dimensions(dataset_code: str) -> None:
    """Print the dimension ids and available codes for a Eurostat dataset."""
    url = f"{EUROSTAT_BASE}/{dataset_code}"
    params = {"format": "JSON", "lang": "EN", "lastTimePeriod": 1}
    response = requests.get(url, params=params, headers=EUROSTAT_HEADERS, timeout=60)
    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text[:300]}")
        return

    data = response.json()
    print(f"\nDimensions of '{dataset_code}':")
    print(f"  Order: {data['id']}")
    for dim_id in data["id"]:
        categories = data["dimension"][dim_id]["category"]
        labels = categories.get("label", {})
        codes = sorted(categories["index"], key=lambda code: categories["index"][code])
        print(f"\n  [{dim_id}]")
        for code in codes[:20]:
            print(f"    {code:30s} -> {labels.get(code, '')}")
        if len(codes) > 20:
            print(f"    ... ({len(codes) - 20} additional codes)")


def download_eurostat_dataset(dataset: dict) -> None:
    output_path = DATA_RAW_DIR / dataset["file_name"]
    print(f"Processing: {dataset['file_name']}...")

    if output_path.exists():
        print(f"  -> File already exists at '{output_path}'. Skipping download.\n")
        return

    response = requests.get(
        f"{EUROSTAT_BASE}/{dataset['dataset_code']}",
        params=dataset["params"],
        headers=EUROSTAT_HEADERS,
        timeout=120,
    )

    if response.status_code == 400:
        raise RuntimeError(
            f"Eurostat 400 Bad Request.\n"
            f"Detail: {response.text}\n"
            f"Hint: uncomment inspect_eurostat_dimensions() to verify the correct "
            f"dimension names and filter codes for this dataset."
        )
    if response.status_code == 413:
        raise RuntimeError("Eurostat 413: asynchronous response. Wait ~2 minutes and re-run the cell.")
    if response.status_code != 200:
        raise RuntimeError(f"HTTP {response.status_code}: {response.text[:500]}")

    data = response.json()
    ids = data["id"]
    dimensions = data["dimension"]
    sizes = data["size"]
    values = data["value"]

    ordered_codes = []
    for dim_id in ids:
        categories = dimensions[dim_id]["category"]
        ordered_codes.append(sorted(categories["index"], key=lambda code: categories["index"][code]))

    records = []
    for combo in itertools.product(*[range(size) for size in sizes]):
        flat_idx = 0
        stride = 1
        for index in range(len(sizes) - 1, -1, -1):
            flat_idx += combo[index] * stride
            stride *= sizes[index]

        value = values.get(str(flat_idx)) if isinstance(values, dict) else (values[flat_idx] if flat_idx < len(values) else None)
        if value is None:
            continue

        row = {ids[index]: ordered_codes[index][combo[index]] for index in range(len(ids))}
        row["OBS_VALUE"] = float(value)
        records.append(row)

    if not records:
        raise ValueError(f"No data after parsing {dataset['file_name']}.")

    df_raw = pd.DataFrame(records)
    df_tidy = (
        df_raw[dataset["keep_columns"]]
        .rename(columns=dataset["rename_map"])
        .sort_values(["Country", "Year"])
        .reset_index(drop=True)
    )

    df_tidy.to_csv(output_path, index=False)
    print(f"  -> Success! {len(df_tidy)} rows saved to '{output_path}'.\n")


for dataset in EUROSTAT_DATASETS:
    download_eurostat_dataset(dataset)

Processing: Eurostat_Energy_Import_Dependency_2000_2025.csv...
  -> File already exists at 'c:\Documenti\UNIMIB\Data Science Lab\PROJECT\Data\Raw\Eurostat_Energy_Import_Dependency_2000_2025.csv'. Skipping download.

Processing: Eurostat_Public_Debt_GDP_2000_2025.csv...
  -> File already exists at 'c:\Documenti\UNIMIB\Data Science Lab\PROJECT\Data\Raw\Eurostat_Public_Debt_GDP_2000_2025.csv'. Skipping download.

